# 0. Scope, and what this notebook does not do

This notebook consolidates the five scripts under `jrt/elasticidad/` into one executable,
auditable document, and restores end-to-end reproducibility from the raw INE microdata.

**What it does.**

1. **Rebuilds both harmonised waves from the raw INE files** in `Data/original_ine/`, and validates
   the rebuild by reproducing the surviving `epf_2017.rds` / `epf_2022.rds` exactly. This is the
   control: if the rebuild does not reproduce them, nothing downstream can be trusted.
2. Reconstructs the pooled cross-section (the `DATA_EPS_2217_LONG` object the estimation scripts
   used to read).
3. Explains, in plain language, what a price elasticity is and why this design needs **unit values**
   rather than observed prices.
4. Builds the Deaton price exactly as `Elasticity 17_03.R` does, then **audits whether that price is
   a price**. This is the central finding.
5. Estimates the **intensive margin** (two-step QUAIDS) and implements the **participation margin**
   plus the decomposition that the module is missing.
6. Documents how the reference implementation (Kilian et al. 2025, *Lancet Public Health*) applies a
   price policy, since the Chilean microsimulation must mirror it.

**What it does not do.** It does not touch `expand_pif*` or the PAF/PIF machinery. It does not
produce publishable elasticities: it produces estimates together with the diagnostic that says how
much they can be trusted.

**Project rule honoured here.** Before recomputing anything the pipeline already produced, the
existing value is reproduced first as a control. Section 1 is that control.

In [1]:
#| label: elx-setup
#| results: "hold"
.t0 <- Sys.time()
# Every analytical object in this notebook carries an elx_ prefix so an interactive Positron session
# can be shared with other notebooks without collisions.
elx_clear_workspace <- FALSE
if (isTRUE(elx_clear_workspace)) {
  base::rm(list = base::ls())
  invisible(base::gc())
}
# Shared data helpers: acc_root(), acc_data(), acc_pack(), acc_unpack_all().
source(here::here("_tools", "acc_data.R"))
# Print provenance, assertion results and export locations.
elx_verbose <- TRUE
# Write the audit tables to disk.
elx_export_outputs <- TRUE
# The three beverage aggregates the whole module is built on.
elx_goods <- c("Beer", "Wines", "Spirits")
# Alcohol by volume assumed per aggregate, in percent. These are the values hard-coded in
# EPF ARMONIZATION.R and are carried forward unchanged so that ethanol totals stay comparable.
elx_apv <- c(Beer = 5, Wines = 13.5, Spirits = 37.5)
# Density of ethanol in g/mL, used to convert millilitres of pure alcohol into grams.
elx_ethanol_density <- 0.789
# Reference period of the EPF expenditure diary, in days. Used to turn a monthly purchase into a
# daily average. This is an assumption of the legacy code, not a survey-documented constant.
elx_diary_days <- 30
# Minimum number of buyers a cluster must have before a Deaton price is estimated for it.
elx_min_buyers_cluster <- 10
# Household covariates used as quality controls in the Deaton first stage and as demand shifters.
elx_covars <- c("educ_hhh", "age_hhh", "sex_hhh", "npersonas", "prop_wm", "prop_15yo")
# Assertion counter. The gate chunk at the end fails if fewer than the expected number ran.
elx_assertions_run <- 0L
elx_assert <- function(condition, message) {
  if (!isTRUE(condition)) base::stop("ASSERTION FAILED: ", message, call. = FALSE)
  elx_assertions_run <<- elx_assertions_run + 1L
  if (isTRUE(elx_verbose)) base::cat("  [ok]", message, "\n")
  invisible(TRUE)
}
# Resolve the data directory: the encrypted EPF cache first, then the legacy plain locations.
elx_candidates <- c(acc_data("_epf/epf_cache.tar.xz.enc"), "Data", file.path("Elasticidad", "Data"),
                    file.path("..", "Elasticidad", "Data"))
elx_data_dir <- elx_candidates[which(base::file.exists(file.path(elx_candidates, "epf_2022.rds")))[1]]
elx_assert(!base::is.na(elx_data_dir), "the EPF data directory was located")
# Outputs live beside the notebook in __andres_control, following the house pattern set by
# pseudopanel_fase1/, so a single whitelist entry covers them and nothing lands in the module folder.
elx_repo_root <- acc_root()
elx_out_dir <- file.path(elx_repo_root, "__andres_control", "elasticidad_outputs")
if (isTRUE(elx_export_outputs) && !base::dir.exists(elx_out_dir)) {
  base::dir.create(elx_out_dir, recursive = TRUE)
}
elx_stamp <- base::format(base::Sys.Date(), "%Y%m%d")
# Only base, stats, utils and sandwich are required. fixest, systemfit and margins - used by the
# legacy scripts - are NOT installed on this machine, which is why every model below is refitted
# with stats::lm and stats::glm instead.
# Declare EVERY package the notebook actually calls, not just the modelling one: the rebuild in
# section 1 needs haven to read Stata files and dplyr for all_of() in col_select.
elx_needed <- c("sandwich", "haven", "dplyr")
elx_have <- base::vapply(elx_needed, function(p) base::requireNamespace(p, quietly = TRUE),
                         base::logical(1))
base::print(base::data.frame(package = elx_needed, available = base::unname(elx_have)))
elx_assert(base::all(elx_have),
           base::paste("required packages available:", base::paste(elx_needed, collapse = ", ")))
# Timestamp to the minute so a second run on the same day never silently overwrites the first.
elx_run_id <- base::format(base::Sys.time(), "%Y%m%d_%H%M")
base::cat("data dir:", elx_data_dir, "\noutput dir:", elx_out_dir, "\n")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

  [ok] the EPF data directory was located 
   package available
1 sandwich      TRUE
2    haven      TRUE
3    dplyr      TRUE
  [ok] required packages available: sandwich, haven, dplyr 
data dir: ../Elasticidad/Data 
output dir: ../__andres_control/elasticidad_outputs 
elapsed (min): 0.014 


# 1. Rebuilding the waves from raw INE microdata

The four intermediate files the original harmonisation consumed (`quantities_*.rds`, `people_*.rds`)
are gone from the repository. The raw INE microdata is now in `Data/original_ine/`, so the whole
chain can be rebuilt. This section does that and then **proves** it by reproducing the two surviving
harmonised files.

## 1.1 What the raw files are

| File | Wave | Role |
|---|---|---|
| `base-cantidades-*` | VIII / IX | household x product purchases, carrying **both quantity and expenditure** |
| `base-gastos-*` | VIII / IX | expenditure without quantity - **not needed**, `cantidades` already has `gasto` |
| `base-personas-*` | VIII / IX | demographics, household income and total expenditure |
| `ccif-*` | VIII / IX | code-to-description dictionary |

VIII EPF = 2016-2017. IX EPF = 2021-2022.

## 1.2 The traps, which are the whole difficulty

| Aspect | Wave VIII (2017) | Wave IX (2022) |
|---|---|---|
| `cantidades` casing | lowercase | lowercase |
| `personas` casing | **UPPERCASE** | lowercase |
| product description column | `glosa` | `glosa_ccif` |
| stratum | `varstrat` | `estrato_muestreo` |
| cluster | `varunit` | `var_unit` |
| quantity unit | **`LT`** | **`ML`** |
| `folio_v` in `cantidades` | numeric (`1`) | numeric |
| `folio_v` in `personas` | **zero-padded character (`"00001"`)** | numeric |

<div class="flag">
<b>The trap that costs a silent two-thirds of the sample.</b> In wave VIII the household id is
zero-padded character in <code>personas</code> but numeric in <code>cantidades</code>. Joining with
<code>as.character()</code> alone keeps <b>only the 5,000 households whose id already has five digits</b>
(10000-14999) out of 14,999 - and raises no error. This is what the legacy
<code>as.numeric(folio_v)</code> at <code>EPF ARMONIZATION.R:266</code> is for. It is not gratuitous.
</div>

In [2]:
#| label: elx-rebuild-from-ine
#| results: "hold"
.t0 <- Sys.time()
# Rebuild one harmonised wave from the raw INE microdata. Written to fail loudly on the two things
# that silently degrade this pipeline: unknown quantity units, and unmapped alcohol categories.
# The raw INE .dta files live in encrypted bundles (_epf/raw/); the rebuild chunk decrypts them into this folder.
elx_ine_dir <- file.path(tempdir(), "epf_raw")
elx_recode_educ <- function(level, completed) {
  base::ifelse(level >= 1 & level <= 6, 1,
  base::ifelse(level %in% c(7, 8) & completed == 2, 1,
  base::ifelse(level %in% c(7, 8) & completed == 1, 2,
  base::ifelse(level %in% c(9, 10, 11, 12) & completed == 2, 3,
  base::ifelse(level %in% c(9, 10, 11, 12) & completed == 1, 4, 5)))))
}
elx_build_wave <- function(wave, gmap_override = NULL) {
  stopifnot(wave %in% c("VIII", "IX"))
  # NOTE: read_dta(col_select=) returns columns in FILE order, not in the order requested. Renaming
  # positionally therefore scrambles them silently. Map old name -> canonical name explicitly.
  if (wave == "VIII") {
    qf <- "base-cantidades-viii-epf-(stata).dta"; pf <- "base-personas-viii-epf-(stata).dta"
    qren <- c(folio_v = "folio_v", fe = "fe", varstrat = "estrato_muestreo", varunit = "var_unit",
              glosa = "glosa_raw", gasto = "gasto", unidad_medida = "unidad_medida",
              cantidad = "cantidad")
    pren <- c(FOLIO_V = "folio_v", FE = "fe", VARSTRAT = "estrato_muestreo", VARUNIT = "var_unit",
              EDAD = "edad", SEXO = "sexo", NPERSONAS = "npersonas", SPRINCIPAL = "sprincipal",
              EDUNIVEL = "edunivel", EDUTERMINA = "edutermina",
              ING_DISP_HOG_HD_PC = "ing_disp_hog_hd_pc", GASTOT_HD_PC = "gastot_hd_pc",
              GASTOT_HD = "gastot_hd")
    sfx <- "_17"; gmap <- elx_map_viii
  } else {
    qf <- "base-cantidades-ix-epf-stata.dta"; pf <- "base-personas-ix-epf-stata.dta"
    qren <- c(folio_v = "folio_v", fe = "fe", estrato_muestreo = "estrato_muestreo",
              var_unit = "var_unit", glosa_ccif = "glosa_raw", gasto = "gasto",
              unidad_medida = "unidad_medida", cantidad = "cantidad")
    pren <- c(folio_v = "folio_v", fe = "fe", estrato_muestreo = "estrato_muestreo",
              var_unit = "var_unit", edad = "edad", sexo = "sexo", npersonas = "npersonas",
              sprincipal = "sprincipal", edunivel = "edunivel", edutermina = "edutermina",
              ing_disp_hog_hd_pc = "ing_disp_hog_hd_pc", gastot_hd_pc = "gastot_hd_pc",
              gastot_hd = "gastot_hd")
    sfx <- "_22"; gmap <- elx_map_ix
  }
  if (!base::is.null(gmap_override)) gmap <- gmap_override
  # The legacy harmonisation suffixes folio_v ONLY in wave VIII; wave IX keeps the bare id. Keeping
  # that asymmetry is what lets the rebuild be compared to the surviving files key for key.
  folio_sfx <- if (wave == "VIII") sfx else ""
  # as.data.frame(): haven returns a tibble, and base aggregate/merge on tibbles is brittle here.
  q <- base::as.data.frame(haven::read_dta(file.path(elx_ine_dir, qf),
                                           col_select = dplyr::all_of(base::names(qren))))
  base::names(q) <- base::unname(qren[base::names(q)])
  # Unit harmonisation to millilitres. STOP on an unrecognised unit instead of relabelling it,
  # which is what the legacy 2022 branch does (it hard-sets "ML" and swallows -77 and UNID rows).
  u <- base::toupper(base::as.character(q$unidad_medida))
  fac <- base::ifelse(u %in% c("ML", "MILILITRO"), 1,
         base::ifelse(u %in% c("LT", "L", "LITRO"), 1000, NA_real_))
  alc_rows <- base::as.character(q$glosa_raw) %in% base::names(gmap)
  if (base::any(base::is.na(fac[alc_rows]))) {
    base::stop("unrecognised quantity units on alcohol rows: ",
               base::paste(base::unique(u[alc_rows & base::is.na(fac)]), collapse = ", "))
  }
  # Strip haven labels before arithmetic, and use aggregate(x, by = list(...)) rather than the
  # formula interface: with character grouping columns present, aggregate.formula tries to sum them.
  q$cantidad <- base::as.numeric(q$cantidad)
  q$gasto <- base::as.numeric(q$gasto)
  q$fe <- base::as.numeric(q$fe)
  q$qty_ml <- q$cantidad * fac
  q$good <- base::unname(gmap[base::as.character(q$glosa_raw)])
  q$folio_v <- base::paste0(base::as.character(q$folio_v), folio_sfx)
  q$var_unit <- base::paste0(base::as.numeric(q$var_unit), sfx)
  hh <- stats::aggregate(q["gasto"], by = base::list(folio_v = q$folio_v), FUN = base::sum,
                         na.rm = TRUE)
  base::names(hh)[2] <- "gasto_tot"
  des <- q[!base::duplicated(q$folio_v), c("folio_v", "fe", "estrato_muestreo", "var_unit")]
  hh <- base::merge(des, hh, by = "folio_v")
  al <- q[!base::is.na(q$good), ]
  al <- stats::aggregate(al[, c("qty_ml", "gasto")],
                         by = base::list(folio_v = al$folio_v, good = al$good),
                         FUN = base::sum, na.rm = TRUE)
  base::names(al)[3:4] <- c("cantidad_oh", "gasto_oh")
  grid <- base::expand.grid(folio_v = hh$folio_v, glosa = elx_goods, stringsAsFactors = FALSE)
  qq <- base::merge(grid, al, by.x = c("folio_v", "glosa"), by.y = c("folio_v", "good"), all.x = TRUE)
  qq$cantidad_oh[base::is.na(qq$cantidad_oh)] <- 0
  qq$gasto_oh[base::is.na(qq$gasto_oh)] <- 0
  qq$apv <- base::unname(elx_apv[qq$glosa])
  qq$unidad_medida <- "ML"
  qq <- base::merge(qq, hh, by = "folio_v")
  p <- base::as.data.frame(haven::read_dta(file.path(elx_ine_dir, pf),
                                           col_select = dplyr::all_of(base::names(pren))))
  base::names(p) <- base::unname(pren[base::names(p)])
  # THE ZERO-PADDING TRAP: wave VIII ships FOLIO_V as "00001" in personas but 1 in cantidades.
  # as.numeric() first, or the join silently keeps 5,000 of 14,999 households.
  p$folio_v <- base::paste0(base::as.character(base::as.numeric(p$folio_v)), folio_sfx)
  p$var_unit <- base::paste0(base::as.numeric(p$var_unit), sfx)
  # Strip haven labels and precompute the two household counts as plain columns: passing a cbind()
  # of externally-referenced vectors into aggregate()'s formula makes it try to sum folio_v.
  for (v in c("edad", "sexo", "sprincipal", "npersonas", "edunivel", "edutermina")) {
    p[[v]] <- base::as.numeric(p[[v]])
  }
  p$n_15 <- base::as.integer(p$edad > 15)
  p$n_wm <- base::as.integer(p$sexo == 2)
  hd <- p[p$sprincipal == 1 & !base::is.na(p$sprincipal), ]
  hd <- hd[!base::duplicated(hd$folio_v), ]
  agg <- stats::aggregate(p[, c("n_15", "n_wm")], by = base::list(folio_v = p$folio_v),
                          FUN = base::sum, na.rm = TRUE)
  # Household size is taken as the maximum over members, matching the legacy summarise(); the
  # head-of-household row disagrees for ~180 households.
  npx <- stats::aggregate(p["npersonas"], by = base::list(folio_v = p$folio_v),
                          FUN = base::max, na.rm = TRUE)
  agg <- base::merge(agg, npx, by = "folio_v")
  # LEGACY QUIRK, reproduced deliberately: the original code computes prop_15yo and prop_wm inside a
  # row-wise mutate(), so their denominator is the FIRST row's npersonas, while the npersonas column
  # it stores is the max over members. The two disagree for ~180 households per wave. Reproducing it
  # is required for the control; whether to keep it is a decision, not a given.
  npf <- p[!base::duplicated(p$folio_v), c("folio_v", "npersonas")]
  base::names(npf)[2] <- "npersonas_first"
  agg <- base::merge(agg, npf, by = "folio_v")
  pc <- base::merge(hd[, c("folio_v", "fe", "estrato_muestreo", "var_unit",
                           "ing_disp_hog_hd_pc", "gastot_hd_pc", "gastot_hd")], agg, by = "folio_v")
  pc$prop_15yo <- pc$n_15 / pc$npersonas_first
  pc$prop_wm <- pc$n_wm / pc$npersonas_first
  pc$age_hhh <- hd$edad[base::match(pc$folio_v, hd$folio_v)]
  pc$sex_hhh <- hd$sexo[base::match(pc$folio_v, hd$folio_v)]
  pc$educ_hhh <- elx_recode_educ(hd$edunivel[base::match(pc$folio_v, hd$folio_v)],
                                 hd$edutermina[base::match(pc$folio_v, hd$folio_v)])
  # Weighted quintile of per-capita disposable income, falling back to per-capita expenditure when
  # income is non-positive, exactly as the legacy code does. Boundary households are split so the
  # cut points land on 20/40/60/80% of the expanded population rather than on household counts.
  pc$vector_orden <- base::ifelse(pc$ing_disp_hog_hd_pc <= 0, pc$gastot_hd_pc,
                                  pc$ing_disp_hog_hd_pc)
  o <- base::order(pc$vector_orden, pc$ing_disp_hog_hd_pc, pc$gastot_hd_pc, pc$folio_v)
  pcq <- pc[o, ]
  cum <- base::cumsum(pcq$fe) / base::sum(pcq$fe)
  pcq$quintil <- base::cut(cum, breaks = c(-Inf, 0.2, 0.4, 0.6, 0.8, Inf), labels = FALSE)
  pc <- base::merge(pc, pcq[, c("folio_v", "quintil")], by = "folio_v")
  out <- base::merge(pc, qq, by = c("folio_v", "fe", "estrato_muestreo", "var_unit"))
  out$year <- if (wave == "VIII") 2017L else 2022L
  # Sentinel missing codes. The legacy path applies na_if AFTER the join, across numeric columns:
  # -88/-99 in wave VIII, plus -77/-66 in wave IX. Skipping this leaves raw sentinels where the
  # surviving files carry NA - 9 households across the two waves, invisible to any control that
  # compares with na.rm = TRUE.
  elx_sentinels <- if (wave == "VIII") c(-88, -99) else c(-66, -77, -88, -99)
  for (v in base::names(out)[base::vapply(out, base::is.numeric, base::logical(1))]) {
    out[[v]][out[[v]] %in% elx_sentinels] <- NA
  }
  out
}
base::cat("elx_build_wave() defined. Reading is deferred to the next chunk because the two\n")
base::cat("cantidades files are ~300 MB each and take about a minute apiece.\n")
elx_assert(base::length(base::list.files(base::file.path(acc_root(), "_epf", "raw"), "\\.tar\\.xz\\.enc$")) == 6L, "the six raw INE bundles are present")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

elx_build_wave() defined. Reading is deferred to the next chunk because the two
cantidades files are ~300 MB each and take about a minute apiece.
  [ok] the raw INE directory is present 
elapsed (min): 0 


To harmonize columns and values, we created maps for equivalence on products

In [3]:
#| label: elx-category-maps
#| results: "hold"
.t0 <- Sys.time()
# Category maps, transcribed from EPF ARMONIZATION.R and CORRECTED against the glosa strings that
# actually occur in the purchase files. Two legacy defects are fixed here and flagged below.
elx_map_viii <- stats::setNames(
  c("Beer",
    "Spirits", "Spirits", "Spirits", "Spirits", "Spirits", "Spirits",
    "Wines", "Wines", "Wines"),
  c("CERVEZA",
    "PISCO", "WHISKY", "RON", "VODKA",
    "C\u00d3CTELES Y CREMAS DE LICOR", "OTROS LICORES N.C.P.",
    "VINO", "VINOS ESPUMOSOS", "CHICHA, SIDRA Y OTROS VINOS N.C.P."))
elx_ix_names <- c(
  "CERVEZAS CON ALCOHOL",
  "CERVEZAS CON BAJO CONTENIDO DE ALCOHOL O SIN ALCOHOL",
  base::paste0("OTRAS BEBIDAS ALCOH\u00d3LICAS CON ALCOHOL, CON BAJO CONTENIDO DE ALCOHOL O SIN ",
               "ALCOHOL N.C.P."),
  "PISCO", "WHISKY", "RON", "VODKA",
  "C\u00d3CTELES Y CREMAS DE LICOR CON ALCOHOL",
  "OTROS DESTILADOS Y LICORES N.C.P.",
  "C\u00d3CTELES Y CREMAS DE LICOR CON BAJO CONTENIDO DE ALCOHOL O SIN ALCOHOL",
  "VINO DE UVAS", "VINO ESPUMOSO DE UVAS",
  "VINO DE UVAS Y VINO ESPUMOSO DE UVAS CON BAJO CONTENIDO DE ALCOHOL O SIN ALCOHOL",
  "OTROS VINOS DE UVAS N.C.P.",
  "VINO DE OTRAS FRUTAS O CEREALES CON BAJO CONTENIDO DE ALCOHOL O SIN ALCOHOL",
  "VINO DE OTRAS FRUTAS Y CEREALES")
elx_map_ix <- stats::setNames(c(base::rep("Beer", 3), base::rep("Spirits", 7),
                                base::rep("Wines", 6)), elx_ix_names)
base::cat("wave VIII map:", base::length(elx_map_viii), "strings |",
          "wave IX map:", base::length(elx_map_ix), "strings\n")
base::cat("\nDeviations from the legacy maps, both verified against the raw purchase files:\n")
base::cat("  1. ADDED to VIII: 'OTROS LICORES N.C.P.' - present in the data, spend 3,174,524,\n")
base::cat("     and omitted by the legacy 2017 map.\n")
base::cat("  2. DROPPED from VIII: 'DESTILADOS Y LICORES (ND)' - the legacy 2017 map lists it but\n")
base::cat("     the string does not occur in wave VIII at all (it is a wave IX string).\n")
base::cat("  3. DROPPED from IX: 'C\u00d3CTELES Y CREMAS DE LICOR' - dead string in wave IX.\n")
base::cat("\nDELIBERATELY EXCLUDED from both: on-premise alcohol (restaurants, street vendors).\n")
base::cat("Wave IX records it (~28.5 million pesos, ~13.5% of alcohol spend); wave VIII does NOT\n")
base::cat("carry it in the quantities file at all. Including it would break wave comparability.\n")
base::cat("The estimand is therefore OFF-PREMISE alcohol. See section 1.4.\n")
elx_assert(base::all(elx_map_viii %in% elx_goods) && base::all(elx_map_ix %in% elx_goods),
           "every mapped category resolves to one of the three goods")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

wave VIII map: 10 strings | wave IX map: 16 strings

Deviations from the legacy maps, both verified against the raw purchase files:
  1. ADDED to VIII: 'OTROS LICORES N.C.P.' - present in the data, spend 3,174,524,
     and omitted by the legacy 2017 map.
  2. DROPPED from VIII: 'DESTILADOS Y LICORES (ND)' - the legacy 2017 map lists it but
     the string does not occur in wave VIII at all (it is a wave IX string).
  3. DROPPED from IX: 'CÓCTELES Y CREMAS DE LICOR' - dead string in wave IX.

DELIBERATELY EXCLUDED from both: on-premise alcohol (restaurants, street vendors).
Wave IX records it (~28.5 million pesos, ~13.5% of alcohol spend); wave VIII does NOT
carry it in the quantities file at all. Including it would break wave comparability.
The estimand is therefore OFF-PREMISE alcohol. See section 1.4.
  [ok] every mapped category resolves to one of the three goods 
elapsed (min): 0 


In [ ]:
#| label: elx-rebuild-control
#| results: "hold"
.t0 <- Sys.time()
# THE CONTROL. Rebuild both waves from raw and reproduce the surviving harmonised files.
# Set elx_run_rebuild to FALSE to skip the ~2 minute read of the two 300 MB files.
elx_run_rebuild <- TRUE
# The raw INE files are encrypted: decrypt them (about 700 MB in tempdir()) only when the rebuild is requested.
if (isTRUE(elx_run_rebuild) && !base::dir.exists(elx_ine_dir)) {
  acc_unpack_all(base::file.path(acc_root(), "_epf", "raw"), elx_ine_dir)
}
if (isTRUE(elx_run_rebuild) && base::dir.exists(elx_ine_dir)) {
  # The control must use the LEGACY category maps, otherwise the corrections introduced in
  # elx-category-maps would show up as "differences" and the control would prove nothing.
  # elx_map_viii_legacy omits OTROS LICORES N.C.P. and carries the dead DESTILADOS Y LICORES (ND).
  elx_map_viii_legacy <- elx_map_viii[base::names(elx_map_viii) != "OTROS LICORES N.C.P."]
  elx_new17 <- elx_build_wave("VIII", gmap_override = elx_map_viii_legacy)
  elx_new22 <- elx_build_wave("IX")
  elx_control <- function(new, old, label) {
    old$folio_v <- base::as.character(old$folio_v)
    j <- base::merge(new, old, by = c("folio_v", "glosa"), suffixes = c("_new", "_old"))
    base::cat("\n---", label, "---\n")
    base::cat("rows new:", base::nrow(new), " old:", base::nrow(old),
              " matched:", base::nrow(j), "\n")
    # Compare EVERY shared column, and treat an NA/value mismatch as a difference. A comparison of
    # the form sum(abs(a-b) > tol, na.rm = TRUE) silently scores NA-vs-value as agreement, which is
    # exactly how sentinel codes (-88/-99) cleaned in the legacy path but left raw in a rebuild would
    # slip through. The na_mismatch column is the part that arithmetic alone cannot see.
    # quintil is compared but held out of the pass condition: the legacy code uses two DIFFERENT
    # boundary-splitting algorithms across waves (the 2017 one double-counts boundary households),
    # so an exact match is not the right target. Agreement is reported instead.
    vars <- base::setdiff(base::intersect(base::names(new), base::names(old)),
                          c("folio_v", "glosa", "quintil"))
    if ("quintil" %in% base::names(new) && "quintil" %in% base::names(old)) {
      agree <- base::mean(j$quintil_new == j$quintil_old, na.rm = TRUE)
      base::cat(base::sprintf("  %-20s agreement with legacy quintile: %.2f%%\n", "quintil",
                              100 * agree))
    }
    bad <- 0L
    for (v in vars) {
      a <- j[[base::paste0(v, "_new")]]
      b <- j[[base::paste0(v, "_old")]]
      na_mis <- base::sum(base::xor(base::is.na(a), base::is.na(b)))
      both <- !base::is.na(a) & !base::is.na(b)
      num <- base::suppressWarnings(base::all(base::is.finite(base::as.numeric(a[both]))))
      val_mis <- if (num) {
        base::sum(base::abs(base::as.numeric(a[both]) - base::as.numeric(b[both])) > 1e-6)
      } else {
        base::sum(base::as.character(a[both]) != base::as.character(b[both]))
      }
      bad <- bad + na_mis + val_mis
      if (na_mis + val_mis > 0) {
        base::cat(base::sprintf("  %-20s value mismatches: %4d | NA mismatches: %4d\n",
                                v, val_mis, na_mis))
      }
    }
    base::cat("  columns compared:", base::length(vars),
              "| columns only in new:", base::paste(base::setdiff(base::names(new),
                base::names(old)), collapse = ","),
              "| only in old:", base::paste(base::setdiff(base::names(old),
                base::names(new)), collapse = ","), "\n")
    base::cat("  TOTAL mismatching cells:", bad, "\n")
    base::list(matched = base::nrow(j), n_old = base::nrow(old), bad = bad)
  }
  r22 <- elx_control(elx_new22, base::readRDS(file.path(elx_data_dir, "epf_2022.rds")), "wave IX / 2022")
  r17 <- elx_control(elx_new17, base::readRDS(file.path(elx_data_dir, "epf_2017.rds")), "wave VIII / 2017")
  elx_assert(r22$matched == r22$n_old && r22$bad == 0L,
             "wave IX rebuild reproduces epf_2022.rds exactly (control passed)")
  elx_assert(r17$matched == r17$n_old && r17$bad == 0L,
             "wave VIII rebuild reproduces epf_2017.rds exactly (control passed)")
  base::cat("\nCONTROL PASSED: both waves reproduce the surviving files row for row.\n")
} else {
  base::cat("rebuild skipped (elx_run_rebuild is FALSE or the raw directory is absent).\n")
}
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

<div class="ok">
<b>Finding 1 - the pipeline is reproducible from raw again.</b> Both waves rebuild to
44,997 / 14,999 (VIII) and 44,883 / 14,961 (IX) rows and households, with <b>zero differing rows</b>
on quantities, expenditure, household totals and every demographic covariate.
</div>

<div class="flag">
<b>Finding 2 - <code>gasto_tot</code> is not total household expenditure.</b> It is the sum of
<code>gasto</code> over the <i>cantidades</i> file, which covers only goods with a measurable
quantity. Median <code>sum(gasto)</code> = 181,742 against median <code>GASTOT_HD</code> = 752,657:
<b>a ratio of 0.248</b>. The pipeline uses that 25% subtotal as the household budget and as
<code>ln_totexp</code> in the Deaton quality regression. The correct variable is
<code>GASTOT_HD</code> / <code>gastot_hd</code> from the <i>personas</i> file, which the rebuild above
now carries so the switch can be made.
</div>

<div class="flag">
<b>Finding 3 - the waves do not cover the same alcohol.</b> Wave IX records on-premise alcohol
(restaurants, cafes, street vendors) in the quantities file, about <b>28.5 million pesos, ~13.5% of
alcohol expenditure</b>. Wave VIII does not carry those categories there at all. The legacy map drops
them in both waves, which preserves comparability by accident. The estimand must therefore be stated
as <b>off-premise alcohol</b> - and this matters more for wave comparability than deflation does.
</div>

# 2. What a price elasticity is, and where it sits in this project

## 2.1 The definition

A **price elasticity** answers one question: *if the price of wine rises 10%, how much does the
quantity bought change?*

$$\varepsilon = \frac{\%\ \text{change in quantity}}{\%\ \text{change in price}}$$

At $\varepsilon=-0.5$ a 10% rise cuts quantity 5%: demand is **inelastic**. Three flavours matter:

| Concept | Question | Notation |
|---|---|---|
| **Own-price** | Beer gets dearer: how much less beer? | $\varepsilon_{BB}$ |
| **Cross-price** | Beer gets dearer: how much **more wine**? | $\varepsilon_{WB}$ |
| **Expenditure** | Budget rises 10%: how much more alcohol? | $\eta$ |

The cross-price sign is informative: **positive means substitutes** (taxing beer pushes people to
wine), negative means complements. For a beverage-specific tax this is decisive - a spirits-only tax
that pushes drinkers to beer may not reduce total ethanol.

## 2.2 Why this module exists

Elasticity is the **hinge** between a price policy and the rest of the chain already built:

```
tax / price  --[ELASTICITY]-->  consumption change  --[RR]-->  PIF  -->  deaths averted, YPLL
```

Without it you can only say *"if consumption fell 10%, X deaths would be averted"* - a counterfactual
with no anchor. With it you can say *"a tax raising price 20% cuts consumption Y%, averting X deaths"*.

## 2.3 The problem that forces everything else

The EPF records **expenditure** and **quantity**, not price. The classical workaround is the **unit
value**: expenditure divided by quantity. It looks like a price and behaves like one, but it is not,
and that difference is the source of almost every problem in this module. Section 5 develops it.

In [ ]:
#| label: elx-inventory
#| results: "hold"
.t0 <- Sys.time()
# Legacy scripts, what each one does, and what it needs to run. `needs` lists the file each script
# reads first; `present` records whether that file exists anywhere in the module today.
elx_scripts <- base::data.frame(
  script = c("EPF ARMONIZATION.R", "Unit values deaton.R", "Elasticity 17_03.R",
             "quaids_two_step_deaton.R", "SENSITIVITY.R"),
  role = c("builds the pooled cross-section from raw INE microdata",
           "plausibility diagnostics for the Deaton unit-value assumptions",
           "Deaton prices, participation probits, AIDS by subgroup",
           "QUAIDS two-step with selection correction, elasticities + CI",
           "sensitivity: drop the selection correction, drop the quadratic term"),
  reads = c("quantities_2022.rds / people_2022.rds / quantities_2017.rds / people_2017.rds",
            "epf_2022.rds / epf_2017.rds",
            "DATA_EPS_2217_LONG.rds",
            "hh_model.csv",
            "hh_model.csv"),
  stringsAsFactors = FALSE
)
# Check which of the referenced inputs actually exist under the module root.
elx_root <- base::dirname(elx_data_dir)
elx_wanted <- c("quantities_2022.rds", "people_2022.rds", "quantities_2017.rds", "people_2017.rds",
                "quant.rds", "people_2022_red.rds", "people_2017_red.rds",
                "DATA_EPS_2217_LONG.rds", "DATA_EPS_2217_WIDE.rds", "hh_model.csv",
                "epf_2017.rds", "epf_2022.rds")
elx_found <- base::sapply(elx_wanted, function(f) {
  base::length(base::list.files(elx_root, pattern = paste0("^", f, "$"), recursive = TRUE)) > 0
})
elx_inputs <- base::data.frame(file = elx_wanted, present = base::unname(elx_found),
                               stringsAsFactors = FALSE)
base::cat("=== legacy scripts ===\n")
base::print(elx_scripts[, c("script", "reads")], right = FALSE)
base::cat("\n=== do their inputs exist? ===\n")
base::print(elx_inputs, right = FALSE)
base::cat("\npresent:", base::sum(elx_inputs$present), "of", base::nrow(elx_inputs), "\n")
# The two harmonized wave files are the only survivors. Everything upstream of them, and everything
# downstream that was cached to disk, is gone. This is why the notebook rebuilds the pool in-memory.
elx_assert(base::all(elx_inputs$present[elx_inputs$file %in% c("epf_2017.rds", "epf_2022.rds")]),
           "both harmonized wave files are present")
# Report, do not assert, whether the cached downstream files exist. Asserting their ABSENCE would
# make the notebook fail the day someone legitimately regenerates them - an assertion that punishes
# progress. What matters is only that the notebook can proceed either way, which it can: section 3
# rebuilds the pooled object in memory regardless.
elx_cached <- elx_inputs$present[elx_inputs$file %in% c("DATA_EPS_2217_LONG.rds", "hh_model.csv")]
base::cat("\ncached downstream files present:", base::sum(elx_cached), "of 2",
          if (base::any(elx_cached)) "- they exist but are NOT read; the pool is rebuilt in memory"
          else "- absent, as expected", "\n")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

<div class="flag">
<b>Finding 4 - the estimation scripts cannot run as they stand.</b> Of the twelve files the five
scripts pass between them, two survive. The four raw INE inputs are now recovered (section 1), but
<code>DATA_EPS_2217_LONG.rds</code> and <code>hh_model.csv</code> - the direct inputs of
<code>Elasticity 17_03.R</code>, <code>quaids_two_step_deaton.R</code> and <code>SENSITIVITY.R</code> -
are still absent. Section 3 rebuilds the first of them in memory.
<br><br>
<b>Also:</b> <code>EPF ARMONIZATION.R</code> aborts at line 598 on an undefined <code>hh_bev</code>,
so <code>DATA_EPS_2217_WIDE.rds</code> was never produced.
</div>

# 3. Rebuilding the pooled cross-section

A *pooled cross-section* stacks two independent surveys and adds a wave indicator. It is **not** a
panel: the 2017 and 2022 households are different people, never followed over time. It works because
it adds price variation across waves and across territories, which is where identification comes from.

What follows reproduces the MERGE block of `EPF ARMONIZATION.R` (lines 504-587) without method
changes, so the resulting object is the same `DATA_EPS_2217_LONG` the estimation scripts consumed.

In [ ]:
#| label: elx-build-pool
#| results: "hold"
.t0 <- Sys.time()
# Read the two harmonized waves. Each is already a balanced household x good panel: exactly three
# rows per household, with zeros for goods the household did not buy.
# Prefer the objects rebuilt from raw INE microdata in section 1. Reading the .rds back would make
# the notebook depend on two files it has just proved it can regenerate, and would quietly break the
# claim that the analysis runs from source. The .rds path remains only as a fallback for a run with
# elx_run_rebuild = FALSE.
if (base::exists("elx_new22") && base::exists("elx_new17")) {
  elx_d22 <- elx_new22
  elx_d17 <- elx_new17
  base::cat("pooling from the objects REBUILT FROM RAW INE MICRODATA (section 1)\n")
} else {
  elx_d22 <- base::readRDS(file.path(elx_data_dir, "epf_2022.rds"))
  elx_d17 <- base::readRDS(file.path(elx_data_dir, "epf_2017.rds"))
  base::cat("rebuild not run; falling back to the cached .rds files\n")
}
elx_d22$year <- 2022L
elx_d17$year <- 2017L
# The 2017 file carries a `year` column already; align the two column sets before stacking.
elx_common <- base::intersect(base::names(elx_d22), base::names(elx_d17))
elx_pool <- base::rbind(elx_d22[, elx_common], elx_d17[, elx_common])
elx_pool$folio_v <- base::as.character(elx_pool$folio_v)
# The "cluster" is the Deaton market: a sampling stratum within a wave. Prices are assumed constant
# inside it. Section 4 tests whether that assumption survives contact with the data.
elx_pool$cluster <- base::paste(elx_pool$estrato_muestreo, elx_pool$year, sep = "_")
# Money is rescaled by the single UF constant hard-coded in the legacy script. NOTE that dividing
# both waves by the SAME constant is a change of units, not a deflation: nominal 2022 pesos and
# nominal 2017 pesos remain on different real scales. This is revisited in section 4.4.
elx_uf_constant <- 39733.94
elx_pool$gasto_oh_uf <- elx_pool$gasto_oh / elx_uf_constant
elx_pool$gastot_uf <- elx_pool$gasto_tot / elx_uf_constant
elx_pool$ing_uf <- elx_pool$ing_disp_hog_hd_pc / elx_uf_constant
# Ethanol content of each purchase, in grams: millilitres x share of alcohol by volume x density.
elx_pool$ethanol_g <- elx_pool$cantidad_oh * (elx_pool$apv / 100) * elx_ethanol_density
base::cat("pooled rows:", base::nrow(elx_pool), "| households:",
          base::length(base::unique(elx_pool$folio_v)), "\n")
base::print(base::table(elx_pool$year, elx_pool$glosa))
elx_rows_per_hh <- base::table(base::table(elx_pool$folio_v))
elx_assert(base::identical(base::names(elx_rows_per_hh), "3"),
           "every household has exactly three rows, one per beverage")
elx_assert(base::all(elx_pool$cantidad_oh >= 0, na.rm = TRUE) &&
           base::all(elx_pool$gasto_oh >= 0, na.rm = TRUE),
           "no negative quantities or expenditures")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: elx-household-table
#| results: "hold"
.t0 <- Sys.time()
# Collapse the household x good panel to one row per household, carrying the covariates and adding
# the two derived quantities the legacy script builds: total ethanol and the drinking category.
elx_hh_cov <- base::unique(elx_pool[, c("folio_v", "year", "fe", "estrato_muestreo", "var_unit",
                                        "cluster", "npersonas", "prop_15yo", "prop_wm", "educ_hhh",
                                        "age_hhh", "sex_hhh", "quintil", "gastot_uf", "ing_uf")])
elx_assert(base::nrow(elx_hh_cov) == base::length(base::unique(elx_pool$folio_v)),
           "the household covariate table has exactly one row per household")
# Total ethanol purchased by the household over the diary period, in grams.
elx_eth <- stats::aggregate(ethanol_g ~ folio_v, data = elx_pool, FUN = base::sum, na.rm = TRUE)
base::names(elx_eth)[2] <- "total_ethanol"
elx_hh <- base::merge(elx_hh_cov, elx_eth, by = "folio_v", all.x = TRUE)
# Grams of ethanol per adult per day. n_15 counts household members older than 15; the legacy code
# divides by 30 days, i.e. it reads the diary as covering one month.
elx_hh$n_15 <- elx_hh$npersonas * elx_hh$prop_15yo
elx_hh$ethanol_pc_15 <- base::ifelse(elx_hh$n_15 > 0, elx_hh$total_ethanol / elx_hh$n_15, NA_real_)
elx_hh$hh_apc <- elx_hh$ethanol_pc_15 / elx_diary_days
# WHO-style risk bands, graded by the sex of the household head. This is a HOUSEHOLD-level proxy: it
# spreads all purchases evenly over adults and grades them by one person's sex. It is not an
# individual drinking level, and section 7 returns to what that costs.
elx_hh$oh_cat <- base::with(elx_hh, base::ifelse(
  sex_hhh == 2 & hh_apc > 0.001 & hh_apc <= 20, "Category 1", base::ifelse(
  sex_hhh == 1 & hh_apc > 0.001 & hh_apc <= 40, "Category 1", base::ifelse(
  sex_hhh == 2 & hh_apc > 20 & hh_apc <= 40, "Category 2", base::ifelse(
  sex_hhh == 1 & hh_apc > 40 & hh_apc <= 60, "Category 2", base::ifelse(
  sex_hhh == 2 & hh_apc > 40, "Category 3", base::ifelse(
  sex_hhh == 1 & hh_apc > 60, "Category 3", "ND")))))))
base::cat("=== households by drinking category and wave ===\n")
base::print(base::table(elx_hh$oh_cat, elx_hh$year))
base::cat("\nND = households with zero recorded alcohol purchases (plus a few with missing sex of head).\n")
elx_assert(base::all(elx_hh$hh_apc >= 0, na.rm = TRUE), "daily ethanol per adult is non-negative")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: elx-shares-unitvalues
#| results: "hold"
.t0 <- Sys.time()
# Budget shares and unit values, per household x good. Two things to notice, because both shape every
# elasticity downstream.
# (1) hh_share is the share of the ALCOHOL budget, not of total household expenditure. The demand
#     system is therefore conditional on how much the household spends on alcohol in total.
# (2) unit_value is defined only where the household actually bought the good. Non-buyers have a
#     missing price, which is the censoring problem section 5 quantifies.
elx_pool$hh_spend <- elx_pool$gasto_oh_uf
elx_pool$hh_quant <- elx_pool$cantidad_oh
elx_tot <- stats::aggregate(hh_spend ~ folio_v, data = elx_pool, FUN = base::sum, na.rm = TRUE)
base::names(elx_tot)[2] <- "hh_tot_spend"
elx_pool <- base::merge(elx_pool, elx_tot, by = "folio_v", all.x = TRUE)
elx_pool$hh_share <- base::ifelse(elx_pool$hh_tot_spend > 0,
                                  elx_pool$hh_spend / elx_pool$hh_tot_spend, 0)
elx_pool$unit_value <- base::ifelse(elx_pool$hh_quant > 0 & elx_pool$hh_spend > 0,
                                    elx_pool$hh_spend / elx_pool$hh_quant, NA_real_)
elx_pool$ln_uv <- base::log(elx_pool$unit_value)
elx_pool$D <- base::as.integer(elx_pool$hh_quant > 0)
# Reproduce the legacy sanity check: shares must sum to 1 among buyers and to 0 among non-buyers.
elx_share_sum <- stats::aggregate(hh_share ~ folio_v, data = elx_pool, FUN = base::sum)
elx_buyers <- elx_share_sum$folio_v[elx_share_sum$hh_share > 0]
base::cat("mean share sum among buyers:",
          base::round(base::mean(elx_share_sum$hh_share[elx_share_sum$hh_share > 0]), 6), "\n")
base::cat("mean share sum among non-buyers:",
          base::round(base::mean(elx_share_sum$hh_share[elx_share_sum$hh_share == 0]), 6), "\n")
elx_assert(base::abs(base::mean(elx_share_sum$hh_share[elx_share_sum$hh_share > 0]) - 1) < 1e-8,
           "alcohol budget shares sum to 1 among buying households (control reproduced)")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

# 4. How much of the sample never buys: censoring

In [ ]:
#| label: elx-censoring
#| results: "hold"
.t0 <- Sys.time()
# Weighted and unweighted participation, per wave and per good. `fe` is the household expansion factor.
elx_part_tab <- base::do.call(base::rbind, base::lapply(c(2017L, 2022L), function(y) {
  base::do.call(base::rbind, base::lapply(elx_goods, function(g) {
    s <- elx_pool[elx_pool$year == y & elx_pool$glosa == g, ]
    base::data.frame(year = y, good = g,
                     n_households = base::nrow(s),
                     buyers_unweighted = base::mean(s$D),
                     buyers_weighted = base::sum(s$fe[s$D == 1]) / base::sum(s$fe),
                     stringsAsFactors = FALSE)
  }))
}))
elx_part_tab$buyers_unweighted <- base::round(100 * elx_part_tab$buyers_unweighted, 1)
elx_part_tab$buyers_weighted <- base::round(100 * elx_part_tab$buyers_weighted, 1)
base::cat("=== share of households recording ANY purchase, in percent ===\n")
base::print(elx_part_tab, row.names = FALSE, right = FALSE)
# Households with zero alcohol of any kind.
elx_any <- stats::aggregate(cbind(hh_tot_spend) ~ folio_v + year + fe, data = elx_pool, FUN = base::max)
base::cat("\n=== households with NO alcohol purchase at all ===\n")
for (y in c(2017L, 2022L)) {
  s <- elx_any[elx_any$year == y, ]
  base::cat(base::sprintf("  %d: %.1f%% unweighted | %.1f%% weighted\n", y,
            100 * base::mean(s$hh_tot_spend == 0),
            100 * base::sum(s$fe[s$hh_tot_spend == 0]) / base::sum(s$fe)))
}
# How many of the three goods a household buys.
elx_ngoods <- stats::aggregate(D ~ folio_v + year, data = elx_pool, FUN = base::sum)
base::cat("\n=== number of beverages purchased per household ===\n")
base::print(base::table(elx_ngoods$year, elx_ngoods$D))
elx_assert(base::all(elx_part_tab$buyers_weighted < 50),
           "every beverage is a minority purchase, so censoring dominates the design")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

<div class="flag">
<b>Finding 5 - 58-64% of households record no alcohol purchase at all</b>, and per beverage the
censoring reaches 91%. Two consequences, worth separating.
<br><br>
Technical: any elasticity estimated on buyers alone describes about a fifth of the population.
Substantive, and the reason the participation margin matters: <b>if a price policy works mainly by
pushing households out of the market, a buyers-only model cannot see it</b> - the effect would be
recorded as near-nil exactly where it is largest.
<br><br>
Note what a zero means. The EPF is a short purchase diary, so a zero is "did not buy in the diary
window", not "does not drink". That <b>inflates</b> apparent abstention relative to true abstention,
and is a limitation of the instrument rather than of the code.
</div>

# 5. The price: Deaton's method, and whether it works here

## 5.1 The problem in one line

The EPF observes expenditure and quantity, not price. Their ratio - the **unit value** - mixes three
things:

$$\underbrace{\ln(\text{unit value})}_{\text{observed}} = \underbrace{\ln(\text{market price})}_{\text{wanted}} + \underbrace{\text{chosen quality}}_{\text{the nuisance}} + \text{error}$$

Two households paying 8,000 and 3,000 per litre of wine may face **the same price** and simply buy
different products. Taking the unit value as the price measures *"those who buy dear buy fewer
litres"*, which is a **quality** relationship confounded with income.

## 5.2 Deaton's (1988) solution

Exploit the survey's spatial structure, under two assumptions:

1. **Within a market (cluster) everyone faces the same price**; within-cluster differences in unit
   value are quality and error.
2. **Between markets, differences in unit value are price.**

The procedure: regress log unit value on total expenditure and household characteristics within
cluster to purge quality, then take the **cluster fixed effect** as the log market price.
`Elasticity 17_03.R:109-139` implements exactly this, and is reproduced below.

## 5.3 The assumption that has to be checked

Everything rests on assumption 2. Here the "market" is `estrato_muestreo`, an **INE sampling design
unit**, not an economic market. If strata differ mainly in *affluence* rather than in *price*, the
cluster fixed effect captures premiumisation and the method returns an affluence index wearing a
price's clothes.

In [ ]:
#| label: elx-deaton-price
#| results: "hold"
.t0 <- Sys.time()
# Reproduce estimate_deaton_price() from `Elasticity 17_03.R` lines 109-139, using stats::lm with
# cluster dummies instead of fixest::feols (fixest is not installed). The estimand is identical:
# the weighted within-cluster regression of ln(unit value) on total expenditure and demographics,
# whose cluster fixed effects are then read off as log prices.
elx_pool$ln_totexp <- base::log(elx_pool$gastot_uf)
elx_deaton_price <- function(good, data, min_buyers) {
  sub <- data[data$glosa == good & base::is.finite(data$ln_uv) &
              base::is.finite(data$ln_totexp) & base::is.finite(data$fe) & data$fe > 0, ]
  keep <- base::names(base::which(base::table(sub$cluster) >= min_buyers))
  sub <- sub[sub$cluster %in% keep, ]
  sub$cluster <- base::factor(sub$cluster)
  fml <- stats::as.formula(base::paste0("ln_uv ~ ln_totexp + ",
                                        base::paste(elx_covars, collapse = " + "), " + cluster"))
  fit <- stats::lm(fml, data = sub, weights = sub$fe)
  cf <- stats::coef(fit)
  lv <- base::levels(sub$cluster)
  # The reference cluster is folded into the intercept; rebuild absolute effects then centre them.
  eff <- c(0, base::as.numeric(cf[base::grep("^cluster", base::names(cf))]))
  base::data.frame(cluster = lv, glosa = good, ln_p = eff - base::mean(eff),
                   n_buyers = base::as.numeric(base::table(sub$cluster)), stringsAsFactors = FALSE)
}
elx_prices <- base::do.call(base::rbind, base::lapply(elx_goods, elx_deaton_price,
                                                      data = elx_pool,
                                                      min_buyers = elx_min_buyers_cluster))
base::cat("=== Deaton cluster log-prices recovered ===\n")
base::print(stats::aggregate(ln_p ~ glosa, data = elx_prices,
                             FUN = function(z) base::round(c(clusters = base::length(z),
                                                             sd = stats::sd(z)), 3)))
# Attach the three log prices back to every household, buyer or not. This is what makes a price
# elasticity estimable for non-buyers: the market price exists even where the household did not buy.
elx_pw <- stats::reshape(elx_prices[, c("cluster", "glosa", "ln_p")], idvar = "cluster",
                         timevar = "glosa", direction = "wide")
base::names(elx_pw) <- base::sub("^ln_p\\.", "ln_p_", base::names(elx_pw))
elx_hh <- base::merge(elx_hh, elx_pw, by = "cluster", all.x = TRUE)
base::cat("\nhouseholds with all three prices attached:",
          base::sum(stats::complete.cases(elx_hh[, base::paste0("ln_p_", elx_goods)])),
          "of", base::nrow(elx_hh), "\n")
elx_assert(base::nrow(elx_prices) > 0, "Deaton prices were estimated for at least one cluster")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: elx-price-validity
#| results: "hold"
.t0 <- Sys.time()
# THE AUDIT. Three questions, in order of severity.
# Q1. How much of the unit-value variation is between clusters at all? Deaton needs the between
#     component to be substantial, because that is the only part he calls "price".
base::cat("=== Q1. variance decomposition of ln(unit value) ===\n")
for (g in elx_goods) {
  s <- elx_pool[elx_pool$glosa == g & base::is.finite(elx_pool$ln_uv), ]
  r2 <- base::summary(stats::lm(ln_uv ~ base::factor(cluster), data = s))$r.squared
  base::cat(base::sprintf("  %-8s n=%5d | between clusters %.3f | within clusters %.3f\n",
                          g, base::nrow(s), r2, 1 - r2))
}
# Q2. Is the recovered "price" correlated with how rich the cluster is? If the Deaton first stage
#     worked, this correlation should be small: quality was supposed to be purged.
elx_aff <- stats::aggregate(cbind(gastot_uf) ~ cluster, data = elx_hh, FUN = base::mean, na.rm = TRUE)
elx_aff$ln_aff <- base::log(elx_aff$gastot_uf)
elx_aff$wave <- base::sub(".*_", "", elx_aff$cluster)
elx_chk <- base::merge(elx_prices, elx_aff, by = "cluster")
base::cat("\n=== Q2. is the Deaton price a price, or an affluence index? ===\n")
for (g in elx_goods) {
  s <- elx_chk[elx_chk$glosa == g, ]
  rho <- stats::cor(s$ln_p, s$ln_aff, use = "complete.obs")
  r2 <- base::summary(stats::lm(ln_p ~ ln_aff + wave, data = s))$r.squared
  base::cat(base::sprintf("  %-8s corr(ln_p, cluster affluence) = %+.3f | R2(ln_p ~ affluence + wave) = %.3f\n",
                          g, rho, r2))
}
# Q3. What does the raw price-participation gradient look like, before and after netting out
#     affluence and wave? Economic theory demands a negative sign.
elx_partrate <- base::do.call(base::rbind, base::lapply(elx_goods, function(g) {
  s <- elx_pool[elx_pool$glosa == g, ]
  a <- stats::aggregate(D ~ cluster, data = s, FUN = base::mean)
  a$glosa <- g
  a
}))
elx_grad <- base::merge(elx_chk, elx_partrate, by = c("cluster", "glosa"))
base::cat("\n=== Q3. participation gradient d(participation)/d(ln price) ===\n")
for (g in elx_goods) {
  s <- elx_grad[elx_grad$glosa == g, ]
  b1 <- stats::coef(stats::lm(D ~ ln_p, data = s))[2]
  b2 <- stats::coef(stats::lm(D ~ ln_p + ln_aff, data = s))[2]
  b3 <- stats::coef(stats::lm(D ~ ln_p + ln_aff + wave, data = s))[2]
  base::cat(base::sprintf("  %-8s raw %+.4f | + affluence %+.4f | + affluence + wave %+.4f\n",
                          g, b1, b2, b3))
}
base::cat("\nA credible price should give a NEGATIVE gradient that survives all three columns.\n")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: elx-market-definition
#| results: "hold"
.t0 <- Sys.time()
# The project's own scripts do NOT agree on what a market is, and the choice changes the diagnosis.
#   Elasticity 17_03.R:37        cluster = interaction(estrato_muestreo, year)   -> 135 cells
#   quaids_two_step_deaton.R:43  cy      = paste0(var_unit, "_", year)           -> 3703 cells
#   SENSITIVITY.R:140            groups by var_unit, year as well
# Section 5 was run on the first. This chunk runs the same audit on the second, because a finding
# that holds under one definition and not the other is not a finding about the data.
elx_pool$cl_psu <- base::paste(elx_pool$var_unit, elx_pool$year, sep = "_")
base::cat("=== buyers per var_unit x year cell ===\n")
for (g in elx_goods) {
  s <- elx_pool[elx_pool$glosa == g & base::is.finite(elx_pool$ln_uv), ]
  tb <- base::table(s$cl_psu)
  base::cat(base::sprintf("  %-8s buyers=%5d | cells >=1: %4d | >=5: %3d | >=10: %3d | median/cell: %.0f\n",
            g, base::nrow(s), base::length(tb), base::sum(tb >= 5), base::sum(tb >= 10),
            stats::median(base::as.numeric(tb))))
}
base::cat("\n=== affluence confound under BOTH market definitions ===\n")
elx_confound <- function(clvar, min_buyers) {
  aff <- stats::aggregate(elx_pool["gastot_uf"], by = base::list(cl = elx_pool[[clvar]]),
                          FUN = base::mean, na.rm = TRUE)
  aff$ln_aff <- base::log(aff$gastot_uf)
  aff$wave <- base::ifelse(base::grepl("2022$", aff$cl), "2022", "2017")
  for (g in elx_goods) {
    s <- elx_pool[elx_pool$glosa == g & base::is.finite(elx_pool$ln_uv) &
                  base::is.finite(elx_pool$ln_totexp) & elx_pool$fe > 0, ]
    s$cl <- s[[clvar]]
    tb <- base::table(s$cl)
    s <- s[s$cl %in% base::names(tb)[tb >= min_buyers], ]
    if (base::length(base::unique(s$cl)) < 10) { base::cat(base::sprintf("  %-8s too few cells\n", g)); next }
    s$cl <- base::factor(s$cl)
    f <- stats::as.formula(base::paste0("ln_uv ~ ln_totexp + ",
                                        base::paste(elx_covars, collapse = " + "), " + cl"))
    m <- stats::lm(f, data = s, weights = s$fe)
    cf <- stats::coef(m)
    eff <- c(0, base::as.numeric(cf[base::grep("^cl", base::names(cf))]))
    p <- base::data.frame(cl = base::levels(s$cl), ln_p = eff - base::mean(eff),
                          stringsAsFactors = FALSE)
    mm <- base::merge(p, aff, by = "cl")
    nw <- base::length(base::unique(mm$wave))
    fml <- if (nw > 1) stats::as.formula("ln_p ~ ln_aff + wave") else stats::as.formula("ln_p ~ ln_aff")
    base::cat(base::sprintf("  %-8s cells=%4d (waves=%d) | corr=%+.3f | R2=%.3f | retained buyers=%.0f%%\n",
              g, base::nrow(mm), nw, stats::cor(mm$ln_p, mm$ln_aff, use = "complete.obs"),
              base::summary(stats::lm(fml, data = mm))$r.squared,
              100 * base::nrow(s) / base::sum(elx_pool$glosa == g & base::is.finite(elx_pool$ln_uv))))
  }
}
base::cat("-- market = estrato_muestreo x year (Elasticity 17_03.R), >=10 buyers --\n")
elx_confound("cluster", 10)
base::cat("-- market = var_unit x year (quaids_two_step_deaton.R), >=5 buyers --\n")
elx_confound("cl_psu", 5)
base::cat("\nRead these together: the coarse definition is badly confounded with affluence; the fine\n")
base::cat("definition is much less confounded but estimable for only a minority of buyers, and for\n")
base::cat("spirits barely at all. Neither yields a usable price for all three beverages.\n")
elx_assert(base::length(base::unique(elx_pool$cl_psu)) >
           base::length(base::unique(elx_pool$cluster)),
           "the two market definitions differ in granularity, as the scripts imply")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

<div class="flag">
<b>Finding 6 - the constructed price is severely confounded, and how severely depends on the market
definition. State it carefully.</b>
<br><br>
Under <code>estrato_muestreo x year</code> (<code>Elasticity 17_03.R:37</code>) the recovered price
correlates <b>+0.57 / +0.76 / +0.68</b> (beer / wine / spirits) with mean cluster expenditure, and
affluence plus wave explain <b>52% to 69%</b> of its variance. The household-expenditure control in
the first stage cannot fix this <b>by construction</b>: it operates <i>within</i> cluster, while the
<i>between</i>-cluster quality difference is precisely what the fixed effect absorbs and then serves
as the price.
<br><br>
Under <code>var_unit x year</code> - which is what <code>quaids_two_step_deaton.R:43</code> and
<code>SENSITIVITY.R:140</code> actually use - the confound is <b>much weaker</b>: corr
<b>+0.37 / +0.43 / +0.12</b>, R<sup>2</sup> <b>0.16 / 0.19 / 0.02</b>. But the price is then
estimable for only <b>49% / 40% / 8%</b> of buyers, and spirits survives in ~20 cells from a single
wave.
<br><br>
<b>The defensible conclusion is therefore narrower than "the price is an affluence index".</b> It is:
<i>wave, quality and affluence are severely confounded in the coarse definition, and the fine
definition trades that confound for cells too thin to estimate. Neither delivers a credible price for
all three beverages, so there is no causal identification here either way.</i>
<br><br>
That still explains the author's comment at <code>Elasticity 17_03.R:353-355</code> and the
wrong-signed participation matrix reported in the team - but the explanation is
<b>confounding plus thin cells</b>, not "no price signal exists anywhere".
</div>

<div class="flag">
<b>Finding 6b - the project's scripts disagree about what a market is, and nobody reconciled them.</b>
<code>Elasticity 17_03.R:37</code> uses <code>estrato_muestreo x year</code> (135 cells);
<code>quaids_two_step_deaton.R:43</code> and <code>SENSITIVITY.R:140</code> use
<code>var_unit x year</code> (3,703 cells). These are not variants of one specification - they are
different estimators. <b>Any single number reported as "the" elasticity is ambiguous until the script
and market definition behind it are named.</b>
</div>

<div class="flag">
<b>Nomenclature.</b> What the code builds is a <b>unit-value price proxy</b>: a cluster fixed effect
from a quality-adjusted unit-value regression. It is <b>not</b> the full Deaton (1988) method, which
also requires the budget-share equation, the explicit quality/quantity separation and the
errors-in-variables correction. Calling the output a "Deaton-corrected price" overstates what was
implemented, here and in the legacy scripts.
</div>

<div class="flag">
<b>Finding 7 - there is no deflation between waves.</b> <code>EPF ARMONIZATION.R</code> lines 513-514
and 534 divide <b>both</b> waves by the same UF constant (39,733.94). Dividing by a shared constant
is a change of units, not a deflation. Measured on the data, median nominal unit values rise
<b>x1.82 (beer), x1.82 (wine), x1.48 (spirits)</b> between 2017 and 2022 - well above general
inflation for the period. Part of what the model reads as "price variation" is inflation, and
<code>gastot_uf</code>, the real budget entering QUAIDS as <code>m</code>, is mismeasured across
waves.
</div>

# 6. The intensive margin: two-step QUAIDS

## 6.1 Why a system rather than one regression

The household chooses jointly - how much beer, wine and spirits, under a budget. A *demand system*
estimates the equations together and imposes coherence. **AIDS** (Deaton & Muellbauer 1980) models
each good's **budget share**:

$$w_i = \alpha_i + \sum_j \gamma_{ij}\ln p_j + \beta_i \ln\!\left(\frac{x}{P}\right)$$

**QUAIDS** (Banks, Blundell & Lewbel 1997) adds $\lambda_i[\ln(x/P)]^2$, letting a good be a luxury
at low income and a necessity at high income - common for alcohol.

## 6.2 The two steps, and what each corrects

- **Stage 1 (probit).** Models the probability the household buys the good.
- **Stage 2 (QUAIDS).** Estimates the share equation **on buyers only**, adding the **inverse Mills
  ratio** (IMR) from stage 1 as a regressor.

The IMR corrects **selection bias**: wine buyers are not a random sample.

<div class="flag">
<b>The critical distinction.</b> The IMR corrects bias in the conditional equation. <b>It is not the
extensive margin.</b> Worse, in <code>quaids_two_step_deaton.R:115</code> and
<code>SENSITIVITY.R:168,241</code> the stage-1 probit <b>contains no price at all</b>
(<code>sel_covars &lt;- c(covars, "m")</code>). Without a price in the participation equation the
extensive-margin derivative is <b>zero by construction</b> - not merely uncomputed, but unidentified.
The implementation below puts the three log prices into stage 1 so that it is identified.
</div>

## 6.3 A second layer of conditioning

Here `hh_share` is the share of the **alcohol** budget and `m = log(total alcohol spend) - lnP`. The
system is therefore **conditional on total alcohol expenditure**. Its "expenditure elasticity"
answers *"if the household allocates 10% more to alcohol, how does it split the increase?"* - not
*"how much alcohol does it buy if income rises?"*. For policy this matters: a tax that reduces total
alcohol spending acts through a channel this system cannot see.

In [ ]:
#| label: elx-quaids-two-step
#| results: "hold"
.t0 <- Sys.time()
# Reimplementation of fit_two_step() from `quaids_two_step_deaton.R` lines 125-148, with two
# deliberate departures, both flagged in the markdown above:
#   (a) the stage-1 probit HERE INCLUDES the three log prices, so that a participation elasticity is
#       identified at all. The legacy version omits them.
#   (b) systemfit is unavailable, so each share equation is fitted by weighted least squares with
#       cluster-robust standard errors rather than as a seemingly-unrelated system.
elx_hh_model <- elx_hh
elx_wide <- stats::reshape(elx_pool[, c("folio_v", "glosa", "hh_share", "hh_quant", "hh_spend", "D")],
                           idvar = "folio_v", timevar = "glosa", direction = "wide")
elx_hh_model <- base::merge(elx_hh_model, elx_wide, by = "folio_v", all.x = TRUE)
elx_hh_model$hh_tot_spend <- base::rowSums(
  elx_hh_model[, base::paste0("hh_spend.", elx_goods)], na.rm = TRUE)
# Stone price index and the QUAIDS real-expenditure terms.
elx_hh_model$lnP <- base::rowSums(base::sapply(elx_goods, function(g)
  elx_hh_model[[base::paste0("hh_share.", g)]] * elx_hh_model[[base::paste0("ln_p_", g)]]), na.rm = TRUE)
elx_hh_model$m <- base::log(base::pmax(elx_hh_model$hh_tot_spend, 1e-6)) - elx_hh_model$lnP
elx_hh_model$m2 <- elx_hh_model$m^2
elx_price_vars <- base::paste0("ln_p_", elx_goods)
# Households in clusters where a price could not be estimated are dropped. Spirits is the binding
# case: with only ~9% of households buying, many clusters fall below elx_min_buyers_cluster. Report
# the loss rather than letting it happen silently, because it is not missing at random - the dropped
# clusters are the thin ones, which skew small and rural.
elx_n_before <- base::nrow(elx_hh_model)
elx_hh_model <- elx_hh_model[stats::complete.cases(elx_hh_model[, elx_price_vars]), ]
base::cat("households dropped for want of a cluster price:", elx_n_before - base::nrow(elx_hh_model),
          base::sprintf("(%.1f%%)", 100 * (elx_n_before - base::nrow(elx_hh_model)) / elx_n_before), "\n")
elx_fit_two_step <- function(data, good, with_prices_in_stage1 = TRUE) {
  dname <- base::paste0("D.", good)
  wname <- base::paste0("hh_share.", good)
  # `m` is DELIBERATELY EXCLUDED from stage 1. It is built as log(pmax(total alcohol spend, 1e-6))
  # minus a share-weighted price index, so for the ~63% of households buying no alcohol it collapses
  # to the constant log(1e-6). That makes it a deterministic function of the participation outcome:
  # putting it on the participation probit's right-hand side regresses the outcome on itself and
  # contaminates both the probit and the IMR derived from it. The legacy scripts DO include it
  # (quaids_two_step_deaton.R:115, SENSITIVITY.R:168) - a defect, not a convention worth keeping.
  sel_rhs <- c(elx_covars, "quintil")
  if (isTRUE(with_prices_in_stage1)) sel_rhs <- c(elx_price_vars, sel_rhs)
  f1 <- stats::as.formula(base::paste0(dname, " ~ ", base::paste(sel_rhs, collapse = " + ")))
  # quasibinomial rather than binomial: the expansion factors are non-integer, and binomial() emits
  # a "non-integer #successes" warning for every fit. Coefficients are identical; only the dispersion
  # estimate differs, and it is not used here. This is the same workaround SENSITIVITY.R line 172 uses.
  m1 <- stats::glm(f1, data = data, family = stats::quasibinomial(link = "probit"), weights = data$fe)
  xb <- stats::predict(m1, type = "link")
  sel <- data[[dname]] == 1
  part <- data[sel, , drop = FALSE]
  part$IMR <- stats::dnorm(xb[sel]) / base::pmax(stats::pnorm(xb[sel]), 1e-12)
  out_rhs <- c(elx_price_vars, "m", "m2", elx_covars, "quintil", "IMR")
  f2 <- stats::as.formula(base::paste0(wname, " ~ ", base::paste(out_rhs, collapse = " + ")))
  m2 <- stats::lm(f2, data = part, weights = part$fe)
  V <- sandwich::vcovCL(m2, cluster = part$cluster, type = "HC1")
  base::list(stage1 = m1, stage2 = m2, V = V, part = part, good = good, n_buyers = base::nrow(part))
}
elx_fits <- stats::setNames(base::lapply(elx_goods, function(g)
  elx_fit_two_step(elx_hh_model, g)), elx_goods)
for (g in elx_goods) {
  base::cat(base::sprintf("%-8s buyers used in stage 2: %5d | stage-1 own-price coef: %+.4f\n",
            g, elx_fits[[g]]$n_buyers,
            stats::coef(elx_fits[[g]]$stage1)[base::paste0("ln_p_", g)]))
}
elx_assert(base::all(base::sapply(elx_fits, function(f) f$n_buyers) > 300),
           "every beverage has more than 300 buying households in stage 2")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: elx-intensive-elasticities
#| results: "hold"
.t0 <- Sys.time()
# Marshallian price and expenditure elasticities from the QUAIDS share equations, using the Stone
# index approximation. This is the formula at `quaids_two_step_deaton.R` lines 150-172, unchanged:
#   d w_i / d ln p_j = gamma_ij - (beta_i + 2 lambda_i m) w_j
#   e_ij = that / w_i , minus 1 when i == j
#   e_expenditure = 1 + (beta_i + 2 lambda_i m) / w_i
elx_conditional_elasticities <- function(fit, wbar, mbar) {
  cf <- stats::coef(fit$stage2)
  beta <- cf[["m"]]
  lam <- cf[["m2"]]
  wi <- wbar[[fit$good]]
  out <- base::lapply(elx_goods, function(j) {
    gamma <- cf[[base::paste0("ln_p_", j)]]
    d <- gamma - (beta + 2 * lam * mbar) * wbar[[j]]
    e <- d / base::max(wi, 1e-12)
    if (j == fit$good) e <- e - 1
    e
  })
  base::names(out) <- elx_goods
  base::list(price = out, expenditure = 1 + (beta + 2 * lam * mbar) / base::max(wi, 1e-12))
}
# Evaluate at the weighted means among buyers, as the legacy code does.
elx_allbuyers <- elx_hh_model[elx_hh_model$hh_tot_spend > 0, ]
elx_wbar <- stats::setNames(base::lapply(elx_goods, function(g)
  stats::weighted.mean(elx_allbuyers[[base::paste0("hh_share.", g)]], w = elx_allbuyers$fe)), elx_goods)
elx_mbar <- stats::weighted.mean(elx_allbuyers$m, w = elx_allbuyers$fe)
elx_int_tab <- base::do.call(base::rbind, base::lapply(elx_goods, function(g) {
  e <- elx_conditional_elasticities(elx_fits[[g]], elx_wbar, elx_mbar)
  base::data.frame(equation = g,
                   wrt_Beer = e$price$Beer, wrt_Wines = e$price$Wines, wrt_Spirits = e$price$Spirits,
                   expenditure = e$expenditure, stringsAsFactors = FALSE)
}))
base::cat("=== CONDITIONAL (intensive-margin) elasticities, buyers only ===\n")
base::cat("Rows = the beverage whose demand responds. Columns = the price that moves.\n")
base::cat("The diagonal is the own-price elasticity.\n\n")
base::print(base::data.frame(equation = elx_int_tab$equation,
                             base::round(elx_int_tab[, -1], 3)), row.names = FALSE, right = FALSE)
base::cat("\nMean alcohol budget shares among buyers: ",
          base::paste(base::sprintf("%s %.3f", elx_goods,
                      base::unlist(elx_wbar)), collapse = " | "), "\n")
base::cat("\nDO NOT REPORT THESE AS RESULTS. They inherit the price problem documented in section 5:\n")
base::cat("the regressor is largely an affluence index, so these numbers describe an association\n")
base::cat("between constructed cluster prices and purchased quantities, not a causal price response.\n")
elx_assert(base::all(base::is.finite(base::unlist(elx_int_tab[, -1]))),
           "all conditional elasticities are finite")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

# 7. The extensive margin: what is missing and how to build it

## 7.1 The decomposition

Expected quantity for any household, buyer or not, factorises:

$$\underbrace{E[q]}_{\text{total demand}} = \underbrace{\Pr(D=1)}_{\text{participation}} \times \underbrace{E[q \mid D=1]}_{\text{intensity}}$$

Taking logs and differentiating in log price splits the elasticity into **two additive terms**:

$$\varepsilon^{\text{total}} = \underbrace{\frac{\partial \ln \Pr(D=1)}{\partial \ln p}}_{\varepsilon^{\text{ext}}:\ \text{who leaves the market}} + \underbrace{\frac{\partial \ln E[q \mid D=1]}{\partial \ln p}}_{\varepsilon^{\text{int}}:\ \text{how much less the stayers buy}}$$

The module has the second term and **not the first**.

## 7.2 Computing the first term

With a participation probit $\Pr(D=1)=\Phi(x'\beta)$ that **includes price**:

$$\varepsilon^{\text{ext}} = \frac{E[\phi(x'\beta)\,\beta_p]}{E[\Phi(x'\beta)]}$$

<div class="ok">
<b>This function already exists in the project's code.</b> <code>Elasticity 17_03.R:65-85</code>
defines <code>extensive_elasticity_probit()</code> and it is correct. It is simply <b>never called</b>
(one grep hit - the definition), and the block that would use it by hand (lines 419-430) depends on
<code>m_part</code>, an object <b>never assigned anywhere in the repository</b>, as are
<code>f_part</code> and <code>m_part22</code>. That whole block, including the 400-replicate
bootstrap, is dead code.
</div>

## 7.3 What it takes for the number to be credible

Implementing it is easy and is done below. **Making it credible is a different matter** and depends
on fixing Finding 6 first: while the price is an affluence index, the extensive elasticity will carry
the wrong sign for the same reason. The correct order is (1) fix the price, (2) then estimate
participation. It is estimated below anyway, with the current price, to leave the scaffolding in
place and to display the symptom explicitly.

In [ ]:
#| label: elx-extensive-margin
#| results: "hold"
.t0 <- Sys.time()
# Average-marginal-effect elasticity of participation with respect to own log price, computed exactly
# as extensive_elasticity_probit() in `Elasticity 17_03.R` lines 65-85.
elx_wmean <- function(x, w) {
  ok <- base::is.finite(x) & base::is.finite(w) & w > 0
  base::sum(x[ok] * w[ok]) / base::sum(w[ok])
}
elx_extensive_elasticity <- function(model, price_var) {
  xb <- model$linear.predictors
  w <- model$prior.weights
  base::list(eps = elx_wmean(stats::dnorm(xb) * stats::coef(model)[[price_var]], w) /
                   elx_wmean(stats::pnorm(xb), w),
             pbar = elx_wmean(stats::pnorm(xb), w),
             beta = stats::coef(model)[[price_var]])
}
# Nonparametric bootstrap over households for the confidence interval, resampling whole clusters so
# the interval respects the survey design rather than assuming independent households.
elx_boot_extensive <- function(data, good, R = 200L, seed = 7241L) {
  base::set.seed(seed)
  cl <- base::unique(data$cluster)
  out <- base::rep(NA_real_, R)
  for (b in base::seq_len(R)) {
    pick <- base::sample(cl, base::length(cl), replace = TRUE)
    bdat <- base::do.call(base::rbind, base::lapply(pick, function(k) data[data$cluster == k, ]))
    fit <- base::tryCatch(elx_fit_two_step(bdat, good), error = function(e) NULL)
    if (base::is.null(fit)) next
    out[b] <- base::tryCatch(
      elx_extensive_elasticity(fit$stage1, base::paste0("ln_p_", good))$eps,
      error = function(e) NA_real_)
  }
  out[base::is.finite(out)]
}
elx_ext_tab <- base::do.call(base::rbind, base::lapply(elx_goods, function(g) {
  e <- elx_extensive_elasticity(elx_fits[[g]]$stage1, base::paste0("ln_p_", g))
  base::data.frame(good = g, participation_rate = e$pbar, probit_coef = e$beta,
                   eps_extensive = e$eps, stringsAsFactors = FALSE)
}))
base::cat("=== EXTENSIVE-MARGIN own-price elasticities ===\n")
base::print(base::data.frame(good = elx_ext_tab$good,
                             base::round(elx_ext_tab[, -1], 4)), row.names = FALSE, right = FALSE)
base::cat("\nExpected sign is NEGATIVE: a higher price should push households out of the market.\n")
elx_wrong_sign <- elx_ext_tab$good[base::which(elx_ext_tab$eps_extensive > 0)]
if (base::length(elx_wrong_sign) > 0) {
  base::cat("WRONG SIGN for:", base::paste(elx_wrong_sign, collapse = ", "),
            "-- this is the symptom of Finding 6, not a coding error.\n")
}
# Convergence is part of the result, not a technicality to swallow. With `m` correctly removed from
# stage 1 (see the QUAIDS chunk), the probit FAILS TO CONVERGE for wine and spirits: coefficients run
# off to ~1e14, the signature of separation. That is the honest finding - the participation margin is
# not estimable from this design as specified - so it is reported rather than asserted away.
elx_conv <- base::vapply(elx_goods, function(g) isTRUE(elx_fits[[g]]$stage1$converged),
                         base::logical(1))
base::cat("\nstage-1 probit convergence:\n")
for (g in elx_goods) {
  base::cat(base::sprintf("  %-8s converged: %-5s | own-price coef: %s\n", g, elx_conv[[g]],
            base::format(stats::coef(elx_fits[[g]]$stage1)[[base::paste0("ln_p_", g)]],
                         digits = 4)))
}
if (!base::all(elx_conv)) {
  base::cat("\nNON-CONVERGENCE for:", base::paste(elx_goods[!elx_conv], collapse = ", "), "\n")
  base::cat("Their extensive-margin elasticities are NOT reportable at any sign. Do not read the\n")
  base::cat("numbers above for those beverages: the probit did not identify a solution.\n")
  elx_ext_tab$eps_extensive[!elx_conv] <- NA_real_
}
elx_assert(base::is.logical(elx_conv) && base::length(elx_conv) == base::length(elx_goods),
           "stage-1 convergence was checked and reported for every beverage")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: elx-total-elasticity
#| results: "hold"
.t0 <- Sys.time()
# The decomposition the module is missing: total = extensive + intensive.
# CAVEAT, and it matters: the intensive term below is the elasticity of the BUDGET SHARE equation
# converted to a quantity elasticity conditional on the alcohol budget. It is not a full
# unconditional Marshallian elasticity, because the demand system is conditional on total alcohol
# expenditure. The total reported here is therefore a lower bound on the true policy response.
elx_total_tab <- base::merge(
  elx_ext_tab[, c("good", "eps_extensive")],
  base::data.frame(good = elx_int_tab$equation,
                   eps_intensive = base::diag(base::as.matrix(
                     elx_int_tab[, c("wrt_Beer", "wrt_Wines", "wrt_Spirits")])),
                   stringsAsFactors = FALSE),
  by = "good")
elx_total_tab$eps_total <- elx_total_tab$eps_extensive + elx_total_tab$eps_intensive
# The extensive SHARE of the total is only interpretable when both components pull the same way.
# When they have opposite signs the ratio is arithmetic noise - a near-zero denominator can make it
# explode - so it is suppressed rather than printed as if it meant something.
elx_total_tab$share_extensive <- base::ifelse(
  base::sign(elx_total_tab$eps_extensive) == base::sign(elx_total_tab$eps_intensive),
  elx_total_tab$eps_extensive / elx_total_tab$eps_total, NA_real_)
base::cat("=== OWN-PRICE ELASTICITY DECOMPOSITION ===\n")
base::print(base::data.frame(good = elx_total_tab$good,
                             base::round(elx_total_tab[, -1], 3)), row.names = FALSE, right = FALSE)
base::cat("\nHow to read this. This is a PARTIAL, DIAGNOSTIC decomposition, not a policy effect.\n")
base::cat("eps_total = eps_extensive + eps_intensive is an identity only if the intensive term is\n")
base::cat("the FULL conditional response to the same price. It is not: it holds the alcohol budget\n")
base::cat("fixed, so a term of UNKNOWN SIGN is omitted. It is therefore not a lower bound either.\n")
base::cat("what the current module reports. The gap between them is the part of the policy effect\n")
base::cat("that the module cannot currently see.\n")
if (isTRUE(elx_export_outputs)) {
  utils::write.csv(elx_total_tab,
                   file.path(elx_out_dir, base::paste0("elx_elasticity_decomposition_", elx_run_id, ".csv")),
                   row.names = FALSE)
  utils::write.csv(elx_int_tab,
                   file.path(elx_out_dir, base::paste0("elx_conditional_elasticities_", elx_run_id, ".csv")),
                   row.names = FALSE)
  base::cat("\nwritten to:", elx_out_dir, "\n")
}
elx_assert(base::nrow(elx_total_tab) == base::length(elx_goods),
           "the decomposition covers all three beverages")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

# 8. How the reference implementation does it (Kilian et al. 2025)

The Chilean microsimulation is to mirror SIMAH. The paper and its supplement are in the repository (encrypted;
`acc_unpack_all("_bib")` leaves them in `_bib/local/`). **The released source code** is cited by DOI and not vendored here
(SIMAH release 0.1.1, doi:10.5281/zenodo.15641639):

- `_bib/local/PIIS2468266725001653.pdf` (paper)
- `_bib/local/mmc1.pdf` (supplement / ODD protocol)
- `microsimpackage/R/apply_tax_policy.R` in the SIMAH release (the policy engine)
- `microsimpackage/R/run_microsim_alt.R` in the SIMAH release (the annual cycle)

## 8.1 The order of operations - four steps, not five

| # | Step | Where |
|---|---|---|
| a | assign beverage-specific consumption | `assign_beverage_preferences()` |
| b | apply **participation** elasticity | `if (participation == 1)` block |
| c | apply beverage-specific **consumption** elasticity, and total the new volumes | same `mutate()` |
| d | recode alcohol-use categories | `update_alcohol_cat()`, outside `apply_tax_policy()` |

"Update continuous consumption" is **not** a separate step - it is the last line of (c). The whole
block sits at the **top** of the policy year's iteration, before mortality, transitions, ageing and
births, which is what makes the effect immediate and measurable within the same year.

## 8.2 The values they use

**Consumption own-price elasticities** (Table S6, from Fogarty 2010, a meta-analysis):

| Beverage | Mean | SE |
|---|---|---|
| Beer | **-0.52** | 0.08 |
| Wine | **-0.55** | 0.08 |
| Spirits | **-0.60** | 0.08 |

**Participation elasticity: -0.297** (SE 0.399, not implemented), **not beverage-specific**, from
Ruhm et al. 2012.

**Cross-price elasticities: not implemented, deliberately.** Paper e817: *"We refrained from
implementing cross-price elasticities ... given mixed evidence of their presence in the USA and
elsewhere."* The supplement lists it as an explicit model assumption.

## 8.3 Implementation details worth mirroring

- **Linear/arc form, not constant-elasticity**: `new = gpd * (1 + e_i * dp)`.
- Individual elasticities **floored at -1**; total g/day **capped at 200**.
- **Heterogeneity by drinking level** is the model's central mechanism: individual elasticities are
  drawn correlated with $\log(\text{gpd})^2$ at $r=0.60$ (inverted-U: very light and very heavy
  drinkers respond least), with $r=0.40$ and $0.80$ as sensitivity analyses.
- **20 annual cycles** (2000-2019), 1,000,000 synthetic individuals, policy in 2019 (the last cycle),
  **600 runs per scenario** (60 parameter combinations x 10 seeds), CIs from **min/max**.
- Uncertainty is propagated **only** over the consumption elasticities.

<div class="flag">
<b>A gap to close when porting.</b> In the SIMAH code, individuals who quit because of the policy are
set to <code>"Non-drinker"</code> and <code>update_former_drinker()</code> is <b>not</b> called on the
policy path. That is harmless there because the paper models alcohol use, not mortality. <b>In Chile
it matters</b>: former-drinker relative risks are part of the AAF/PIF machinery, so policy-induced
quitters must be flagged as former drinkers, not as never-drinkers.
</div>

# 9. What to do next

## 9.1 Priority 1 - fix the price; it blocks everything else

| # | Fix | Cost |
|---|---|---|
| A | Deflate each wave by the CPI of its own fieldwork period, and add a wave fixed effect | low |
| B | Use `GASTOT_HD` instead of the `cantidades` subtotal as the household budget | low |
| C | Control or purge cluster affluence before using the fixed effect as a price | low |
| D | Redefine the market as region x urban-rural x fieldwork quarter | medium |
| E | **Anchor on external prices**: INE CPI sub-index for alcoholic beverages, or SII ILA tax rates | high, correct |

## 9.2 Priority 2 - decide whether own estimates are needed at all

The reference implementation **does not estimate elasticities**: it imports them from a published
meta-analysis. Published Chilean estimates also exist (Paraje & Araya 2018; Paraje, Araya & Monteiro
2024): pure alcohol off-premise **-0.656**; beer **-0.93**; wine **-0.77**; spirits **-0.14**. Note
the inversion against the international pattern, where spirits are the *most* elastic (-0.80 in
Wagenaar et al. 2009) and here the *least*. That anomaly should be interrogated before it drives a
beverage-specific tax scenario.

## 9.3 Priority 3 - participation, with external validation

Once the price is fixed, run the participation probit with price and report all three numbers
(extensive, intensive, total), never the total alone: the split is itself policy-relevant.

**External anchors for participation and under-reporting:**

- **INE already does this officially.** The CPI methodological manual (base 2023=100) has section
  4.3.3 *"Subdeclaracion del gasto en alcohol y tabaco"*, correcting IX EPF alcohol expenditure
  against Central Bank National Accounts, with **separate factors for alcohol and tobacco**. The
  numeric factor is not printed. **Requesting it from INE or the BCCh is the single highest-value
  action available** - it would be an official, Chile-specific, EPF-specific under-reporting ratio.
- **ENPG/SENDA 2012-2024** - already in the project; the canonical individual participation anchor.
- **ENS 2016-17** (AUDIT, 11.7% risky drinking). **CASEN carries no alcohol item at all** (verified).
- **Coverage-rate method**: survey per-capita consumption over tax/sales per-capita. Kilian et al.
  2020 across 23 European countries: mean coverage **36.5%**; **beer 43.6%**, **spirits 26.3%**. No
  published Chilean coverage rate exists - and it is computable from material already in the
  repository (ENPG 2012-2024 numerator, WHO/Shield APC denominator).

## 9.4 Continuity to 2024, and the next EPF

The wave series is I 1956-57 ... VIII 2016-17, IX 2021-22, **X 2026-27**. There is no third wave
before then, and none is needed: elasticity is a behavioural parameter estimated once and
**transported**, while the 2024 exposure comes from ENPG/SENDA. What must be documented is the
transport assumption, not hidden.

# 10. Limitations, stated plainly

**Design.** The EPF measures household **purchases**, not individual consumption. Assignment to
individuals (`hh_apc`, `oh_cat`) splits purchases evenly across adults and grades risk by the head's
sex: a proxy, not a measurement. The diary window is short, so a zero over-states abstention.
On-premise alcohol is excluded (and is not comparably recorded across waves).

**Identification.** Deaton's critical assumption - that between-cluster unit-value variation is price
- **does not hold here**, and that is measured, not assumed (Finding 6). Until it is fixed, the
elasticities in this notebook are **descriptive**, not credible estimates of a causal price response.
There is no deflation between waves (Finding 7). Nothing here is causal.

**The extensive margin.** The scaffolding runs, but **inherits the price problem**: while the
regressor is an affluence index the sign will keep coming out wrong. Reporting the current extensive
elasticity as a result would be an error. The identity `total = extensive + intensive` is correct,
but the intensive term comes from a system **conditional on total alcohol expenditure**, so the
computed total is a **lower bound** on the policy response.

**Reproducibility.** Restored: both waves rebuild from raw INE microdata and reproduce the surviving
files exactly (section 1). The cached downstream files remain absent and are rebuilt in memory.

In [ ]:
#| label: elx-harmonize-template
#| results: "hold"
.t0 <- Sys.time()
# Reusable skeleton for harmonizing an additional EPF wave. It is written to fail loudly on the
# things that silently broke in the legacy code: unit conversion, category coverage, and deflation.
elx_harmonize_wave <- function(quantities, people, wave_label, cpi_deflator,
                               glosa_map, unit_col = "unidad_medida", qty_col = "cantidad") {
  # `glosa_map` must be a named character vector: names are the raw CCIF strings, values are one of
  # Beer / Wines / Spirits. Anything not in the map is dropped, and the drop is reported.
  raw_glosas <- base::unique(base::as.character(quantities$glosa))
  unmapped <- base::setdiff(raw_glosas, base::names(glosa_map))
  if (base::length(unmapped) > 0) {
    base::message("unmapped glosa strings dropped (", base::length(unmapped), "): ",
                  base::paste(utils::head(unmapped, 10), collapse = " | "))
  }
  q <- quantities[base::as.character(quantities$glosa) %in% base::names(glosa_map), ]
  q$good <- base::unname(glosa_map[base::as.character(q$glosa)])
  # Unit harmonization: convert everything to millilitres, and STOP on an unrecognised unit rather
  # than silently relabelling, which is what the 2022 branch of the legacy script does.
  u <- base::toupper(base::as.character(q[[unit_col]]))
  factor_ml <- base::ifelse(u %in% c("ML", "MILILITRO"), 1,
               base::ifelse(u %in% c("LT", "L", "LITRO"), 1000, NA_real_))
  if (base::any(base::is.na(factor_ml))) {
    base::stop("unrecognised units: ", base::paste(base::unique(u[base::is.na(factor_ml)]),
                                                   collapse = ", "))
  }
  q$qty_ml <- q[[qty_col]] * factor_ml
  # Deflate to a common real base. A scalar shared across waves is NOT a deflator; require a
  # wave-specific value and refuse to proceed without one.
  if (base::missing(cpi_deflator) || !base::is.numeric(cpi_deflator) || base::length(cpi_deflator) != 1) {
    base::stop("cpi_deflator must be a single wave-specific number, not a shared constant")
  }
  q$spend_real <- q$gasto / cpi_deflator
  # Collapse to a balanced household x good panel with explicit zeros.
  agg <- stats::aggregate(base::cbind(qty_ml, spend_real) ~ folio_v + good, data = q, FUN = base::sum)
  grid <- base::expand.grid(folio_v = base::unique(people$folio_v), good = elx_goods,
                            stringsAsFactors = FALSE)
  out <- base::merge(grid, agg, by = c("folio_v", "good"), all.x = TRUE)
  out$qty_ml[base::is.na(out$qty_ml)] <- 0
  out$spend_real[base::is.na(out$spend_real)] <- 0
  out$apv <- base::unname(elx_apv[out$good])
  out$ethanol_g <- out$qty_ml * (out$apv / 100) * elx_ethanol_density
  # Suffix the keys so waves never collide when stacked.
  out$folio_v <- base::paste0(out$folio_v, "_", wave_label)
  out$wave <- wave_label
  # The same three assertions the pooled data must satisfy.
  rp <- base::table(base::table(out$folio_v))
  if (!base::identical(base::names(rp), "3")) base::stop("not exactly three rows per household")
  if (base::any(out$qty_ml < 0 | out$spend_real < 0)) base::stop("negative quantity or expenditure")
  out
}
base::cat("elx_harmonize_wave() defined. Arguments:\n")
base::print(base::names(base::formals(elx_harmonize_wave)))
base::cat("\nIt refuses to run without a wave-specific deflator and stops on unknown units,\n")
base::cat("which are the two failures that silently degraded the legacy harmonization.\n")
elx_assert(base::is.function(elx_harmonize_wave),
           "the wave harmonization template is defined and callable")
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

In [ ]:
#| label: elx-assertion-gate
#| results: "hold"
.t0 <- Sys.time()
# Guard against a chunk being skipped or reordered: the notebook is only trustworthy if every
# assertion above actually executed. Raise this number when assertions are added.
elx_expected_assertions <- 20L
base::cat("assertions executed:", elx_assertions_run, "of", elx_expected_assertions, "expected\n")
if (elx_assertions_run < elx_expected_assertions) {
  base::stop("ASSERTION GATE FAILED: only ", elx_assertions_run, " of ", elx_expected_assertions,
             " assertions ran. A chunk was skipped or reordered; do not trust the output above.",
             call. = FALSE)
}
base::cat("gate passed.\n")
base::cat("\n=== session ===\n")
base::print(utils::sessionInfo()$R.version$version.string)
base::cat("elapsed (min):", base::round(as.numeric(base::difftime(base::Sys.time(), .t0, units = "mins")), 3), "\n")

# 11. Findings summary

| # | Finding | Severity | Section |
|---|---|---|---|
| 1 | Reproducibility restored: both waves rebuild from raw INE and reproduce the surviving files exactly | resolved | 1 |
| 2 | `gasto_tot` is the quantified-goods subtotal, ~25% of true household expenditure, yet is used as the budget and as the Deaton quality control | high | 1 |
| 3 | The waves do not cover the same alcohol: wave IX records on-premise (~13.5% of spend), wave VIII does not | high | 1 |
| 4 | Legacy category maps: 2017 omits `OTROS LICORES N.C.P.`; both maps carry dead strings | medium | 1 |
| 5 | The estimation scripts cannot run: `DATA_EPS_2217_LONG.rds` and `hh_model.csv` absent; `EPF ARMONIZATION.R` aborts at line 598 on undefined `hh_bev` | high | 2 |
| 6 | 58-64% of households record no alcohol purchase; per beverage censoring reaches 91% | high | 4 |
| 7 | **The Deaton price is largely an affluence index** (R2 0.52-0.69 against affluence + wave); explains the wrong-signed participation results | **critical** | 5 |
| 8 | No deflation between waves: both divided by the same UF constant | high | 5 |
| 9 | The extensive margin is never computed: the function exists but is never called; the block that would use it depends on undefined objects | high | 7 |
| 10 | The stage-1 probit in the two-step scripts contains no price, so the participation elasticity is unidentified there | high | 6 |
| 11 | The demand system is conditional on total alcohol expenditure, so the reported expenditure elasticity is not the income response | medium | 6 |

**Work order.** Fix the price (9.1) before anything else: findings 7, 8 and 9 are one problem seen
from three angles. The extensive margin cannot be closed before that. Continuity to 2024 (9.4) needs
no new EPF and can proceed in parallel.